<a href="https://colab.research.google.com/github/kamurathegit/marketing_datacleaning/blob/main/marketing_data_ipnyb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
df = pd.read_csv('/content/marketing_campaign_data_messy.csv')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   object 
 1   Campaign_Name  2020 non-null   object 
 2   Start_Date     2020 non-null   object 
 3   End_Date       2020 non-null   object 
 4   Channel        1919 non-null   object 
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   object 
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   object 
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 189.5+ KB
None


In [ ]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


In [ ]:
#header cleaning
print(df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
print("fixed")
print(df.columns.to_list())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
fixed
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [ ]:
#type conversion

dirty_spend = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend,['campaign_id','spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print('fixed')
print(df.loc[dirty_spend,['campaign_id','spend']].head(3))


   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
fixed
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [ ]:
#categorical typos
print(df['channel'].unique())

cleanup_map = {
  'Facebok' :  'Facebook',
  'Insta_gram' : 'Instagram',
  'Tik_Tok' : 'TikTok',
  'Gogle' : 'Google',
  'E-mail' : 'Email',
  'N/A' : np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)
print('fixed')
print(df['channel'].unique())


['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
fixed
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan 'Google']


In [ ]:
#handlinlg mixed booleans
print(df['active'].unique())

df['active'] = df['active'].replace(
    {
        'Y' : 'True',
        '1' : 'True',
        'Yes' : 'True',
        'True' : 'True',
        'N' : 'False',
        'No' : 'False',
        '0' : 'False',
        'False' : 'False',

    }
)
print('fixed')
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
fixed
['True' 'False']


In [ ]:
#date parsing
df['start_date'] = pd.to_datetime(df['start_date'], errors = 'coerce')
print('fixed')
print(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'], errors = 'coerce')
print('fixed')
print(df['end_date'])

fixed
0      2023-11-24
1      2023-05-06
2      2023-12-13
3             NaT
4      2023-04-22
          ...    
2015   2023-10-31
2016   2023-09-01
2017   2023-02-09
2018   2023-03-30
2019   2023-06-26
Name: start_date, Length: 2020, dtype: datetime64[ns]
fixed
0      2023-12-13
1      2023-05-12
2      2023-12-20
3      2023-11-03
4      2023-04-23
          ...    
2015   2023-11-13
2016   2023-09-26
2017   2023-02-21
2018   2023-04-27
2019   2023-07-09
Name: end_date, Length: 2020, dtype: datetime64[ns]


In [ ]:
df = (df.loc[:,~df.columns.duplicated(keep='first')])

In [ ]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


In [ ]:
#logical integrity (clicks vs impressions)

impossible = df['clicks'] > df['impressions']
print(df.loc[impossible,['campaign_id','clicks','impressions']].head(3))

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


In [ ]:
#logical integrity(time travel)
time_travel = df['start_date'] > df['end_date']
print(df.loc[time_travel,['campaign_id', 'start_date', 'end_date']].head(3))

df.loc[time_travel, 'end_date'] = df.loc[time_travel, 'start_date'] + pd.Timedelta(days = 1)
print('fixed')
print(df.loc[time_travel,['campaign_id', 'start_date', 'end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
fixed
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-07
54   CMP-00055 2023-09-01 2023-09-02
71   CMP-00072 2023-02-01 2023-02-02


In [ ]:
#handling outliers
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

outliers = df['spend'] > upper_limit
print(df.loc[outliers,['campaign_id','spend']])

df.loc[outliers, 'spend'] = upper_limit
print('fixed')
print(df.loc[outliers,['campaign_id','spend']])




     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
1718   CMP-01719  500000.00
1754   CMP-01755  500000.00
1781   CMP-01782  500000.00
fixed
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375
1718   CMP-01719  8603.5375
1754   CMP-01755  8603.5375
1781   CMP-01782  8603.5375


In [ ]:
#string passing
print(df['campaign_name'].head(3))

df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')
print('fixed')
print(df['season'].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
fixed
0    Summer
1    Launch
2    Winter
Name: season, dtype: object


In [ ]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter
